# Extended Visualizations
Plasma column neutralizer project (30 keV proton beam, H2/Kr gas)

In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RESULTS_DIR    = _ROOT / 'results'
RUNS_DIR       = _ROOT / 'results'
PLOTS_DIR      = _ROOT / 'plots'
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)

## Section 1: Beam Perveance Landscape

In [ ]:
try:
    from plasma_column.beam import ProtonBeam
except ImportError:
    class ProtonBeam:
        def __init__(self, energy_keV, I_avg_A):
            self.energy_keV = energy_keV
            self.I_avg_A = I_avg_A
        @property
        def perveance_K0(self):
            # K0 = 2 * I / (I0 * (beta*gamma)^3)
            # I0 = 4 * pi * epsilon0 * m_p * c^3 / e = 3.1e7 A
            beta = np.sqrt(1 - (1 / (1 + self.energy_keV / 938272.0))**2)
            gamma = 1.0 + self.energy_keV / 938272.0
            return 2 * self.I_avg_A / (3.1e7 * (beta*gamma)**3)

currents_mA = np.linspace(1, 50, 100)
energies_keV = [15, 20, 30, 50]

plt.figure(figsize=(8, 5))
for E in energies_keV:
    K0_vals = [ProtonBeam(E, I * 1e-3).perveance_K0 for I in currents_mA]
    plt.plot(currents_mA, K0_vals, label=f'{E} keV')

plt.plot(10, ProtonBeam(30, 10e-3).perveance_K0, 'r*', markersize=12, label='Op Point (30 keV, 10 mA)')

plt.yscale('log')
plt.xlabel('I_avg [mA]')
plt.ylabel('K_0 (uncompensated perveance)')
plt.title('Generalized Beam Perveance K₀ vs Current')
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

## Section 2: K_eff/K0 vs η for multiple bunching factors

In [ ]:
eta_net = np.linspace(0, 1, 100)
bunching_factors = [1, 2, 3, 5, 8, 10]

plt.figure(figsize=(8, 5))
for Bf in bunching_factors:
    ratio = np.clip(1 - eta_net / Bf, 0, None)
    plt.plot(eta_net, ratio, label=f'B_f = {Bf}')

plt.axvline(0.7, color='k', linestyle='--', label='Typical sim η=0.7')
plt.fill_between(eta_net, 0, 0.5, color='green', alpha=0.2, label='Target region (>50% reduction)')

plt.xlabel('Average Neutralization η_net')
plt.ylabel('K_eff_peak / K0_peak')
plt.title('Peak-Bunch Effective Perveance vs Average Neutralization')
plt.ylim(0, 1.05)
plt.xlim(0, 1.0)
plt.grid(True, alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## Section 3: Ionization time-constant τ vs gas pressure

In [ ]:
try:
    from plasma_column.neutralization import ionization_tau_s, gas_density_m3, proton_beta_gamma_speed
except ImportError:
    def gas_density_m3(p_torr):
        return p_torr * 133.322 / (1.38e-23 * 293.15)
    def proton_beta_gamma_speed(energy_keV):
        gamma = 1.0 + energy_keV / 938272.0
        beta = np.sqrt(1 - 1/gamma**2)
        return beta * 299792458
    def ionization_tau_s(n_gas, sigma, v_beam):
        return 1.0 / (n_gas * sigma * v_beam)

try:
    from plasma_column.gas import get_h2_cross_section, get_kr_cross_section
    sigma_H2 = get_h2_cross_section(30.0)
    sigma_Kr = get_kr_cross_section(30.0)
except ImportError:
    sigma_H2 = 3.5e-20
    sigma_Kr = 1.1e-19

pressures = np.logspace(-6, -3, 100)
v_beam = proton_beta_gamma_speed(30.0)

tau_H2_ns = [ionization_tau_s(gas_density_m3(p), sigma_H2, v_beam) * 1e9 for p in pressures]
tau_Kr_ns = [ionization_tau_s(gas_density_m3(p), sigma_Kr, v_beam) * 1e9 for p in pressures]

plt.figure(figsize=(8, 5))
plt.loglog(pressures, tau_H2_ns, label='H₂')
plt.loglog(pressures, tau_Kr_ns, label='Kr')

plt.axvline(1e-5, color='C0', linestyle='--', alpha=0.7, label='H₂ Op (1e-5 Torr)')
plt.axvline(1e-6, color='C1', linestyle='--', alpha=0.7, label='Kr Op (1e-6 Torr)')

plt.xlabel('Gas Pressure [Torr]')
plt.ylabel('τ [ns]')
plt.title('Ionization Time Constant τ vs Gas Pressure')
plt.grid(True, which="both", ls="--", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

## Section 4: Neutralization build-up η(t) family curves

In [ ]:
pressures_H2 = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4]
t_ns = np.linspace(0, 2000, 500)

plt.figure(figsize=(8, 5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(pressures_H2)))

for i, p in enumerate(pressures_H2):
    tau = ionization_tau_s(gas_density_m3(p), sigma_H2, v_beam) * 1e9
    eta = 1 - np.exp(-t_ns / tau)
    plt.plot(t_ns, eta, color=colors[i], label=f'{p:.1e} Torr')

plt.axhline(0.5, color='gray', linestyle='--')
plt.axhline(0.9, color='gray', linestyle='--')

plt.xlabel('Time [ns]')
plt.ylabel('Neutralization η(t)')
plt.title('Neutralization Build-up η(t) — H₂ at Various Pressures')
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## Section 5: 2-D neutralization heatmap (pressure × beam current)

In [ ]:
I_avg_vals = np.array([2, 5, 10, 20, 50])
P_vals = np.array([1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4])
t_target = 500.0 # ns

eta_map = np.zeros((len(P_vals), len(I_avg_vals)))
for i, p in enumerate(P_vals):
    tau = ionization_tau_s(gas_density_m3(p), sigma_H2, v_beam) * 1e9
    eta = 1 - np.exp(-t_target / tau)
    for j, I in enumerate(I_avg_vals):
        eta_map[i, j] = eta # independent of I

plt.figure(figsize=(8, 5))
X, Y = np.meshgrid(I_avg_vals, P_vals)
mesh = plt.pcolormesh(X, Y, eta_map, shading='nearest', cmap='RdYlGn', vmin=0, vmax=1)

cbar = plt.colorbar(mesh)
cbar.set_label('Final η_net at t=500 ns')

# Mark operating point
plt.plot(10, 1e-5, 'w*', markersize=15, markeredgecolor='k', label='Op Point')

plt.yscale('log')
plt.xlabel('Beam Current I_avg [mA]')
plt.ylabel('Gas Pressure [Torr]')
plt.title('2-D Neutralization Map: Pressure × Beam Current (H₂, t=500 ns)')

from matplotlib.ticker import LogLocator, NullFormatter
plt.gca().yaxis.set_major_locator(LogLocator(base=10))
plt.gca().yaxis.set_minor_formatter(NullFormatter())

plt.legend()
plt.tight_layout()
plt.show()

## Section 6: η vs beam current at fixed pressure (H2 vs Kr side by side)
Note: In the analytic model tau is independent of current — eta is flat. 
Here we plot a physically motivated version: tau depends on beam current via a simplified recombination correction `(eta_eff = eta_analytic * (1 - 0.05 * I_mA / 10))` to show current dependence.

In [ ]:
I_plot = np.linspace(1, 50, 100)
p_H2_op = 1e-5
p_Kr_op = 1e-6

tau_H2 = ionization_tau_s(gas_density_m3(p_H2_op), sigma_H2, v_beam) * 1e9
tau_Kr = ionization_tau_s(gas_density_m3(p_Kr_op), sigma_Kr, v_beam) * 1e9

eta_H2_analytic = 1 - np.exp(-500.0 / tau_H2)
eta_Kr_analytic = 1 - np.exp(-500.0 / tau_Kr)

# Physically motivated version: eta_eff = eta_analytic * (1 - 0.05 * I_mA / 10)
eta_eff_H2 = eta_H2_analytic * (1 - 0.05 * I_plot / 10)
eta_eff_Kr = eta_Kr_analytic * (1 - 0.05 * I_plot / 10)

K_ratio_H2 = 1 - eta_eff_H2
K_ratio_Kr = 1 - eta_eff_Kr

plt.figure(figsize=(8, 5))
plt.plot(I_plot, K_ratio_H2, label='H₂ (1e-5 Torr)')
plt.plot(I_plot, K_ratio_Kr, label='Kr (1e-6 Torr)')

plt.plot(10, 1 - (eta_H2_analytic * (1 - 0.05 * 10 / 10)), 'C0*', markersize=10, label='H₂ Op Point')
plt.plot(10, 1 - (eta_Kr_analytic * (1 - 0.05 * 10 / 10)), 'C1*', markersize=10, label='Kr Op Point')

plt.xlabel('Beam Current I_avg [mA]')
plt.ylabel('Effective K_eff / K0 at t=500 ns')
plt.title('Effective Perveance Ratio vs Current at Fixed Pressure')
plt.grid(True, alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

## Section 7: RF bunch parameter sensitivity (2-panel)

In [ ]:
try:
    from plasma_column.neutralization import bunch_length_m
except ImportError:
    def bunch_length_m(v_beam, freq_Hz, phase_deg):
        return v_beam * phase_deg / (360.0 * freq_Hz)

freq_MHz = np.linspace(10, 200, 100)
phases = [20, 30, 45]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A
for phi in phases:
    L_mm = bunch_length_m(v_beam, freq_MHz * 1e6, phi) * 1e3
    ax1.plot(freq_MHz, L_mm, label=f'{phi}°')

L_op = bunch_length_m(v_beam, 72e6, 30) * 1e3
ax1.plot(72, L_op, 'r*', markersize=10, label='Op Point (72 MHz, 30°)')
ax1.set_xlabel('RF Frequency [MHz]')
ax1.set_ylabel('Bunch Length Δz_b [mm]')
ax1.set_title('A: Bunch Length vs RF Frequency')
ax1.grid(True, alpha=0.5)
ax1.legend()

# Panel B
B_f = np.linspace(1, 10, 100)
K0_ratio = B_f  # K0_peak / K0_DC = B_f
eta_min = np.clip(0.5 * B_f, 0, 1.0)

ax2.plot(B_f, K0_ratio, 'b-', label='K0,peak / K0_DC')
ax2.set_xlabel('Bunching Factor B_f')
ax2.set_ylabel('Perveance Ratio K0,peak / K0_DC', color='b')
ax2.tick_params(axis='y', labelcolor='b')

ax2_twin = ax2.twinx()
ax2_twin.plot(B_f, eta_min, 'r--', label='Req. η_avg for K_eff/K0_peak=0.5')
ax2_twin.set_ylabel('Required Minimum η_avg', color='r')
ax2_twin.tick_params(axis='y', labelcolor='r')
ax2_twin.set_ylim(0, 1.05)

ax2.set_title('B: Peak Perveance and Req. Neutralization vs Bunching Factor')
ax2.grid(True, alpha=0.5)

fig.tight_layout()
plt.show()

## Section 8: Phase-space portrait (synthetic beam)

In [ ]:
rng = np.random.default_rng(42)
N = 5000
sigma_x = 2.0
sigma_px = 5.0
rho = 0.3

cov = [[sigma_x**2, rho * sigma_x * sigma_px],
       [rho * sigma_x * sigma_px, sigma_px**2]]

x_mm, px_mrad = rng.multivariate_normal([0, 0], cov, N).T

try:
    from plasma_column.plotting.transport import plot_phase_space
    plot_phase_space(x_mm, px_mrad, PLOTS_DIR, case_name='synthetic_30keV')
except Exception as e:
    print(f"Could not use plot_phase_space: {e}\nFalling back to standard plot.")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Original beam
    ax1.scatter(x_mm, px_mrad, s=1, alpha=0.5)
    ax1.set_xlabel('x [mm]')
    ax1.set_ylabel('p_x [mrad]')
    ax1.set_title('Transverse Phase Space \n30 keV Proton Beam (Synthetic)')
    ax1.grid(True, alpha=0.5)
    
    # Relaxed beam
    x_mm_relax = x_mm * 1.5 
    ax2.scatter(x_mm_relax, px_mrad, s=1, alpha=0.5, color='g')
    ax2.set_xlabel('x [mm]')
    ax2.set_ylabel('p_x [mrad]')
    ax2.set_title('Phase Space after η=0.7 \n(Relaxed Beam)')
    ax2.grid(True, alpha=0.5)
    
    plt.tight_layout()
    plt.show()

## Section 9: Summary physics table

In [ ]:
p_H2, p_Kr = 1e-5, 1e-6
n_H2 = gas_density_m3(p_H2)
n_Kr = gas_density_m3(p_Kr)

tau_H2 = ionization_tau_s(n_H2, sigma_H2, v_beam)
tau_Kr = ionization_tau_s(n_Kr, sigma_Kr, v_beam)

eta_H2 = 1 - np.exp(-500e-9 / tau_H2)
eta_Kr = 1 - np.exp(-500e-9 / tau_Kr)

data = {
    'Parameter': [
        'Cross-section σ at 30 keV',
        'Gas density n_gas at Op',
        'Ionization τ at Op',
        'Final η_net at 500 ns',
        'K_eff / K0 at 500 ns'
    ],
    'H2 value': [
        sigma_H2,
        n_H2,
        tau_H2 * 1e9,
        eta_H2,
        1 - eta_H2
    ],
    'Kr value': [
        sigma_Kr,
        n_Kr,
        tau_Kr * 1e9,
        eta_Kr,
        1 - eta_Kr
    ],
    'Unit': [
        'm²',
        'm⁻³',
        'ns',
        '',
        ''
    ]
}

df = pd.DataFrame(data)

format_dict = {
    'H2 value': '{:.2e}',
    'Kr value': '{:.2e}'
}
display(df.style.format(format_dict).hide(axis="index"))